In [1]:
import boto3
import botocore
import os

In [2]:
session = boto3.Session(profile_name='default')

In [3]:
S3_CLIENT = session.client('s3')

## Automatizador de subida de archivos a Amazon S3

In [6]:
# Automatizador de subida de archivos a Amazon S3
import os
from botocore.exceptions import ClientError

"""
Objetivo
Crear un script en Python que suba automáticamente archivos locales a un bucket.
Servicios involucrados
Amazon S3
boto3
Funcionalidad del proyecto
El script debe:
Subir un archivo
Listar archivos en el bucket
Descargar un archivo
"""
def existe_archivo(archivo):
    return os.path.exists(archivo)

def listar_archivos_de_bucket(bucket):
    try:
        response = S3_CLIENT.list_objects_v2(
            Bucket=bucket
        )
    except ClientError as error:
        print(f"Ocurrio un error : {error}")
        return
    

    contents = response['Contents']
    name = response['Name']

    print(f"Contenido de Bucket : {name}")
    print(f"----------------------------")

    for item in contents:
        key = item['Key']
        lastModified = item['LastModified']

        print(f"OBJETO : {key}  -> Ultima modificacion: {lastModified}")

def subir_archivo_a_bucket(archivo, bucket, key=None):
    
    if not existe_archivo(archivo):
        print("El archivo no existe! Intenta de nuevo")
        return
    
    if key is None:
        key  = archivo

    try:
        response = S3_CLIENT.put_object(
            Body=archivo,
            Bucket=bucket,
            Key=key
        )

        eTag = response['ETag']

        print("Se ha subido el archivo exitosamente")
        print(f"ETag : {eTag}\n")

    except ClientError as error:
        print(f"Ocurrio un error : {error}")
        return


In [28]:
archivo_prueba='archivo_prueba.txt'
bucket_name='m30-codespace-sample-bucket-401989331654'

print(f"Subiendo archivo : {archivo_prueba}")

subir_archivo_a_bucket(
    archivo=archivo_prueba, 
    bucket=bucket_name
)

print(f"Listando objetos en bucket : {bucket_name}")

listar_archivos_de_bucket(bucket=bucket_name)

Subiendo archivo : archivo_prueba.txt
Se ha subido el archivo exitosamente
ETag : "872c76819acc3fb9b0922bc1b4d34284"
Listando objetos en bucket : m30-codespace-sample-bucket-401989331654
Contenido de Bucket : m30-codespace-sample-bucket-401989331654
----------------------------
OBJETO : archivo_prueba.txt  -> Ultima modificacion: 2026-04-09 05:18:30+00:00


## Sistema automático de Backups a S3

Crear un script que **haga backup automático de una carpeta local** hacia S3.

**Servicios involucrados**
- Amazon S3

**Funcionalidad**
El programa debe:

- Escanear una carpeta
- Subir todos los archivos
- Crear una carpeta con fecha

In [11]:
list(os.walk("carpeta-prueba/prueba2"))[0]

('carpeta-prueba/prueba2', [], ['texto4.txt'])

In [34]:
from datetime import datetime
root_path + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

'carpeta-prueba2026-04-09_15-12-22'

In [9]:
from datetime import datetime

def recuperar_elementos_recursivos_de_carpeta(carpeta):
        root_path, dirnames, filenames = list(os.walk("carpeta-prueba"))[0]
        return root_path, dirnames, filenames

def elemento_es_directorio(carpeta):
        return os.path.isdir(carpeta)

def generar_nombre_de_carpeta_unico(carpeta):
        return carpeta + "-" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

def respaldar_carpeta_a_bucket_S3(carpeta, bucket):
        
        if not elemento_es_directorio(carpeta):
                print("La carpeta introducida no es valida! Intenta de nuevo")
                return
    
        root_path, _, filenames = recuperar_elementos_recursivos_de_carpeta(carpeta)

        unique_root_path = generar_nombre_de_carpeta_unico(root_path)

        for filename in filenames:      
                
                print(f"Subiendo archivo : {filename}")

                subir_archivo_a_bucket(
                archivo=os.path.join(root_path, filename), 
                bucket=bucket,
                key=os.path.join(unique_root_path, filename)
                )

In [10]:
respaldar_carpeta_a_bucket_S3("carpeta-prueba", "bucket-test-lmsf-1982")

Subiendo archivo : texto3.txt
Se ha subido el archivo exitosamente
ETag : "ade747e14441cd5c4340f7b0f14665e4"

Subiendo archivo : texto2.txt
Se ha subido el archivo exitosamente
ETag : "4353095f192338f3a00c5cf952bf02ed"

Subiendo archivo : texto1.txt
Se ha subido el archivo exitosamente
ETag : "080597d5efbaf458ee6113f69dfbfb56"

